# Real-time Person Detection with YOLOv11 (Version 1)

This is the simplified version focusing only on detecting people without cumulative counting.

In [ ]:
import cv2
import torch
from ultralytics import YOLO
import time

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load pre-trained YOLO11n model
model = YOLO("yolo11n.pt")
model.to(device)
print("Model loaded.")

In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open webcam.")
else:
    prev_time = 0
    
    while True:
        success, frame = cap.read()
        if not success:
            break

        # Run detection - classes=[0] counts only 'person'
        results = model(frame, stream=True, verbose=False, classes=[0])

        # Process results
        for r in results:
            annotated_frame = r.plot()

        # FPS Calculation
        curr_time = time.time()
        fps = 1 / (curr_time - prev_time) if (curr_time - prev_time) > 0 else 0
        prev_time = curr_time
        
        cv2.putText(annotated_frame, f"FPS: {int(fps)}", (20, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.imshow("YOLOv11 Real-time Person Detection", annotated_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Resources released.")